# Vergleich: Pandas, Dask, PySpark

pandas --> „Alles auf meinem Rechner“

Dask --> „Pandas auf mehreren Partitionen“

PySpark --> "Ähnlich wie Dask, aber kommt aus einer anderen Welt (Apache Spark, Java)"

| Aspekt             | pandas                          | Dask                                         | PySpark                        |
| ------------------ | ------------------------------- | -------------------------------------------- | ------------------------------ |
| Hauptzweck         | Datenanalyse auf einem Rechner  | pandas-artige Verarbeitung parallelisieren   | verteilte Datenverarbeitung    |
| Typischer Einsatz  | kleine bis mittlere Datenmengen | größere Daten auf einem Rechner oder Cluster | sehr große Daten, Cluster, ETL |
| API                | Python/pandas                   | stark an pandas angelehnt                    | eigene DataFrame-API + SQL     |
| Parallelisierung   | begrenzt                        | ja                                           | ja                             |
| Mehrere Rechner    | nein                            | ja                                           | ja                             |
| Daten > RAM        | schwierig                       | ja                                           | ja                             |
| Lazy Evaluation    | überwiegend nein                | ja                                           | ja                             |
| Installation/Setup | sehr einfach                    | relativ einfach                              | etwas schwergewichtiger        |
| Ökosystem          | NumPy, SciPy, scikit-learn      | Python Data Science                          | Spark/Hadoop/Data Lakes        |
| JVM nötig          | nein                            | nein                                         | ja                             |


Dask und Spark sehen sich konzeptionell erstaunlich ähnlich

Das finde ich für das Verständnis besonders wichtig:

                 pandas              Dask                 Spark
    
    Data        DataFrame        DataFrame             DataFrame
                                  │                      │
                                  ├─ Partition           ├─ Partition
                                  ├─ Partition           ├─ Partition
                                  └─ Partition           └─ Partition
    
    Execution   direkt           lazy                   lazy
    
    Parallel    wenig            ja                     ja
    
    Cluster     nein             optional               zentraler Use Case

Der große Unterschied ist die Herkunft.

Dask kommt aus der Python-Welt

Im Zentrum stehen:

    Python
      ↓
    NumPy
      ↓
    pandas
      ↓
    Dask

Deshalb fühlt sich Dask für einen Python-Nutzer sehr natürlich an.

Dask DataFrame ist tatsächlich aus pandas-DataFrames aufgebaut und versucht, eine vertraute pandas-artige API bereitzustellen.

Spark kommt aus der Big-Data-/Cluster-Welt

Historisch eher:

    Big Data
       ↓
    Hadoop
       ↓
    Spark
       ↓
    Spark SQL
       ↓
    PySpark

Python ist dabei eine von mehreren Schnittstellen zu Spark.

# PySpark Mini Crash Course

PySpark ist im Grunde die Python-Schnittstelle zu Apache Spark. Damit kannst du große Datenmengen verteilt auf mehreren CPU-Kernen oder sogar auf vielen Rechnern verarbeiten, ohne die verteilte Ausführung selbst programmieren zu müssen.

Für eine Einführung ist es hilfreich, PySpark mit pandas zu vergleichen:

- pandas: typischerweise Datenverarbeitung auf einem Rechner, im Hauptspeicher
- PySpark: Daten werden in Partitionen zerlegt und können verteilt verarbeitet werden
- Beide arbeiten häufig tabellarisch mit Zeilen und Spalten
- PySpark führt viele Operationen erst dann wirklich aus, wenn ein Ergebnis benötigt wird

Der zentrale Datentyp in modernem PySpark ist der DataFrame.

Ein sehr kleines Beispiel:

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkExample") \
    .getOrCreate()

data = [
    ("Anna", 23),
    ("Ben", 31),
    ("Clara", 27)
]

df = spark.createDataFrame(data, ["name", "age"])

df.show()

+-----+---+
| name|age|
+-----+---+
| Anna| 23|
|  Ben| 31|
|Clara| 27|
+-----+---+



In [7]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

SparkSession ist dabei normalerweise dein Einstiegspunkt in Spark.

Du kannst einen DataFrame ähnlich wie in SQL bearbeiten:

In [8]:
df.select("name", "age").show()

+-----+---+
| name|age|
+-----+---+
| Anna| 23|
|  Ben| 31|
|Clara| 27|
+-----+---+



In [12]:
# select() erzeuge ein neues DataFrame
# Das aber erstmal nicht angezeigt wird
df.select("age", "name")

DataFrame[age: bigint, name: string]

In [18]:
id(df)

134938461654864

In [19]:
id(df.select("age", "name"))

134938447802736

In [20]:
# Mit .show() auf dem neuen DataFrame können wir es auch sehen
df.select("age", "name").show()

+---+-----+
|age| name|
+---+-----+
| 23| Anna|
| 31|  Ben|
| 27|Clara|
+---+-----+



In [16]:
# Es gibt auch ein columns Attribut wie bei Pandas DataFrames
df.columns

['name', 'age']

In [72]:
# Das Tabellenschema sagt uns welche Spalten es gibt und
# den Datentyp
df.schema

NameError: name 'df' is not defined

In [25]:
# Beispiel: Daten filtern mit booleschem Ausdruck
df.filter((df.age > 25) & (df.age < 35)).show()

+-----+---+
| name|age|
+-----+---+
|  Ben| 31|
|Clara| 27|
+-----+---+



In [24]:
# Beispiel: Daten filtern wie vorher, aber mit String
df.filter("age > 25 and age < 35").show()

+-----+---+
| name|age|
+-----+---+
|  Ben| 31|
|Clara| 27|
+-----+---+



Ein wichtiger Unterschied zu pandas ist, dass PySpark stark mit Expressions arbeitet. Wenn du zum Beispiel eine neue Spalte erzeugen möchtest:

In [26]:
from pyspark.sql.functions import col

df2 = df.withColumn(
    "age_next_year",
    col("age") + 1
)

df2.show()

+-----+---+-------------+
| name|age|age_next_year|
+-----+---+-------------+
| Anna| 23|           24|
|  Ben| 31|           32|
|Clara| 27|           28|
+-----+---+-------------+



Typische Operationen in PySpark sind:

    df.select(...)
    df.filter(...)
    df.withColumn(...)
    df.groupBy(...)
    df.orderBy(...)
    df.join(...)

Zum Beispiel Gruppierung:

In [27]:
data = [
    ("Berlin", 20),
    ("Berlin", 30),
    ("Hamburg", 40),
    ("Hamburg", 20)
]

df = spark.createDataFrame(data, ["city", "value"])

df.groupBy("city").avg("value").show()

+-------+----------+
|   city|avg(value)|
+-------+----------+
| Berlin|      25.0|
|Hamburg|      30.0|
+-------+----------+



In [28]:
df.show()

+-------+-----+
|   city|value|
+-------+-----+
| Berlin|   20|
| Berlin|   30|
|Hamburg|   40|
|Hamburg|   20|
+-------+-----+



Ein besonders wichtiges Konzept bei Spark ist Lazy Evaluation. Angenommen, du schreibst:

In [29]:
result = (
    df
    .filter(col("value") > 10)
    .select("city", "value")
)

Dann führt Spark diese Berechnung noch nicht unbedingt aus. Spark merkt sich zunächst nur:

Erst bei einer sogenannten Action wird tatsächlich gerechnet.

Zum Beispiel:

In [30]:
result.show()

+-------+-----+
|   city|value|
+-------+-----+
| Berlin|   20|
| Berlin|   30|
|Hamburg|   40|
|Hamburg|   20|
+-------+-----+



In [31]:
result.count()

4

Man unterscheidet deshalb grob zwischen Transformations und Actions.

Transformations erzeugen einen neuen DataFrame:

    df.filter(...)
    df.select(...)
    df.withColumn(...)
    df.groupBy(...)

Actions lösen die tatsächliche Berechnung aus:

    df.show()
    df.count()
    df.collect()
    df.write...

Das ist eines der wichtigsten Spark-Konzepte.

Warum macht Spark das? Weil Spark dadurch mehrere Operationen gemeinsam optimieren kann.
Aus:

    df.filter(...)
      .select(...)
      .groupBy(...)

kann Spark zunächst einen Ausführungsplan erzeugen und anschließend entscheiden, wie dieser möglichst effizient ausgeführt wird.

Ein weiteres zentrales Konzept sind Partitionen. Stell dir einen DataFrame mit 1 Milliarde Zeilen vor:

    DataFrame
    
    Partition 1   → Worker/Core 1
    Partition 2   → Worker/Core 2
    Partition 3   → Worker/Core 3
    Partition 4   → Worker/Core 4
    ...

Dadurch können viele Daten parallel verarbeitet werden.

PySpark-Code sieht trotzdem relativ gewöhnlich aus:

    df.filter(col("age") > 30)
    
Spark kümmert sich darum, diese Operation auf die Partitionen zu verteilen.

Ein typisches Programm könnte daher so aussehen:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

spark = SparkSession.builder \
    .appName("StudentAnalysis") \
    .getOrCreate()

df = spark.read.csv(
    "students.csv",
    header=True,
    inferSchema=True
)

result = (
    df
    .filter(col("age") >= 18)
    .groupBy("course")
    .agg(avg("grade").alias("average_grade"))
    .orderBy("average_grade")
)

result.show()

In [ ]:
Konzeptionell passiert:

    CSV
     ↓
    DataFrame
     ↓
    filter age >= 18
     ↓
    group by course
     ↓
    average grade
     ↓
    sort
     ↓
    show()

Neben der DataFrame-API kannst du Spark auch direkt mit SQL verwenden:    

In [ ]:
df.createOrReplaceTempView("students")

spark.sql("""
    SELECT course, AVG(grade) AS average_grade
    FROM students
    WHERE age >= 18
    GROUP BY course
""").show()

Das Schöne ist: DataFrame-API und Spark SQL verwenden intern weitgehend dieselbe Ausführungsengine.

Ein wichtiger Punkt: PySpark bedeutet nicht, dass die eigentliche Datenverarbeitung hauptsächlich in Python stattfindet. Python dient stark als Schnittstelle. Spark selbst läuft auf der JVM und seine Query Engine plant und optimiert die Berechnungen. Deshalb sollte man möglichst die eingebauten Spark-Funktionen verwenden:

In [32]:
from pyspark.sql.functions import col, avg, sum, max

# Apache Spark, PySpark, Spark SQL

    Apache Spark
    │
    ├── Spark Core
    │
    ├── Spark SQL
    │   └── DataFrame API
    │
    ├── Structured Streaming
    │
    └── weitere Komponenten
    
    PySpark = Python-Schnittstelle zu Apache Spark

Der wichtigste Punkt ist also:

- Spark ist das gesamte verteilte Rechensystem.

- Spark SQL ist ein Teil von Spark, spezialisiert auf strukturierte/tabellarische Daten und SQL-artige Verarbeitung.

- PySpark ist die Python-API, mit der du Spark aus Python steuerst.

Ein konkretes Beispiel macht es klarer.

Wenn du schreibst:

    from pyspark.sql import SparkSession    
    spark = SparkSession.builder.getOrCreate()

dann benutzt du PySpark, also die Python-Schnittstelle.

Wenn du dann schreibst:

    df = spark.read.parquet("data/sales")

arbeitest du mit einem Spark DataFrame.

Dieser DataFrame gehört funktional zu Spark SQL.

Das heißt:

    Python-Code
       ↓
    PySpark
       ↓
    Spark SQL / DataFrame Engine
       ↓
    Spark-Ausführung
       ↓
    CPU-Kerne / Worker / Cluster

Merke:

    Spark = das verteilte Rechensystem
    Spark SQL = die Tabellen-/SQL-Schicht von Spark
    PySpark = der Python-Zugang zu Spark

# Hadoop?

    Big-Data-Ökosystem
    
    Hadoop
    ├── HDFS          ← verteiltes Dateisystem
    ├── YARN          ← Ressourcen-/Cluster-Manager
    └── MapReduce     ← älteres Rechenmodell
    
    Spark
    ├── Spark SQL
    ├── DataFrames
    ├── Streaming
    └── ML

Hadoop war ursprünglich ein Gesamtpaket für verteilte Speicherung und Verarbeitung großer Datenmengen.

Spark ist vor allem eine Rechen-Engine für verteilte Datenverarbeitung.

Historisch war Hadoop stark geprägt von MapReduce:

    Daten
      ↓
    Map
      ↓
    Zwischenergebnisse auf Platte
      ↓
    Reduce
      ↓
    Ergebnis

Spark wurde unter anderem deshalb populär, weil viele Verarbeitungsschritte effizienter verkettet werden können und nicht ständig Zwischenergebnisse auf Platte geschrieben werden müssen.

Ein wichtiger Teil von Hadoop ist HDFS:

    HDFS = Hadoop Distributed File System

Dabei wird eine große Datei über mehrere Rechner verteilt gespeichert:

    große Datei
    
    Block 1 → Rechner A
    Block 2 → Rechner B
    Block 3 → Rechner C

und typischerweise zusätzlich repliziert, damit der Ausfall eines Rechners nicht gleich Datenverlust bedeutet.

Spark kann sehr gut Daten aus HDFS lesen:

    HDFS
      ↓
    Spark
      ↓
    DataFrame
      ↓
    filter / join / groupBy

Aber: Spark braucht Hadoop nicht zwingend.

Spark kann auch Daten lesen aus:

    lokalem Dateisystem
    S3
    Azure Blob Storage
    Google Cloud Storage
    Datenbanken
    Parquet-Dateien
    ...

Ebenso braucht Spark nicht zwingend YARN. Es kann zum Beispiel laufen:

    lokal
    auf Kubernetes
    mit Spark Standalone
    auf YARN

# Installation

Zuerst prüfst du, ob du Java hast:

In [2]:
!java --version

openjdk 21.0.12 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)


Spark läuft auf der JVM, daher brauchst du ein installiertes Java. Falls noch keines vorhanden ist, kannst du auf Ubuntu z. B. installieren:

In [3]:
#sudo apt update
#sudo apt install openjdk-17-jdk

Für eine lokale Installation reicht normalerweise aus das pyspark Python Paket zu installieren. Du musst Apache Spark nicht zusätzlich manuell herunterladen; das pyspark-Paket bringt die nötigen Spark-Komponenten für die lokale Nutzung mit.

Für die aktuelle PySpark-Version 4.2.0 brauchst du mindestens Java 17. Unterstützt werden aktuell Java 17, 21 und 25. Die PySpark-Dokumentation nennt ausdrücklich „Java 17 or later“ und verlangt, dass JAVA_HOME korrekt gesetzt ist.

In [4]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 99.7 MB/s eta 0:00:00m eta 0:00:010:00:01
  Preparing metadata (setup.py) ... done
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  DEPRECATION: Building 'pyspark' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pyspark'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for pyspark: filename=pyspark-4.2.0-py2.py3-none-any.whl size=450798675 sha256=f1ff287ba1ba88bd78d6c6c226cff71440359ca025120806343d1cdbb1d4547e
  Stored in directory: /home/juebrauer/.cache/pip/wheels/a7/94/11/60b8a9f7f3b008ee04da64b003784e8bbaf1d5473f05b412d

Damit arbeitest du zunächst im sogenannten *Local Mode*: Spark läuft vollständig auf deinem eigenen Rechner. Trotzdem lernst du dabei dieselbe DataFrame-API und praktisch dieselben Konzepte, die später auch auf einem Spark-Cluster verwendet werden.

# Erstes Vergleichsbeispiel

## Datensatz erzeugen

100 Millionen Einträge, Verkaufsdaten, aufgeteilt in mehrere Parquet-Dateien

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

N_ROWS = 100_000_000
CHUNK_SIZE = 2_000_000

output_dir = Path("data/sales")
output_dir.mkdir(exist_ok=True)

rng = np.random.default_rng(42)

categories = np.array([
    "electronics",
    "books",
    "clothing",
    "food",
    "sports"
])

countries = np.array([
    "DE",
    "FR",
    "IT",
    "ES",
    "NL"
])

for i, start in enumerate(range(0, N_ROWS, CHUNK_SIZE)):

    n = min(CHUNK_SIZE, N_ROWS - start)

    df = pd.DataFrame({
        "customer_id": rng.integers(1, 1_000_000, n),
        "product_id": rng.integers(1, 100_000, n),
        "category": rng.choice(categories, n),
        "country": rng.choice(countries, n),
        "quantity": rng.integers(1, 10, n),
        "price": rng.uniform(1, 500, n),
        "year": rng.choice([2023, 2024, 2025, 2026], n)
    })

    df.to_parquet(
        output_dir / f"part-{i:03d}.parquet",
        index=False
    )

    print(f"{start+n:,}", end=" ")

2,000,000 4,000,000 6,000,000 8,000,000 10,000,000 12,000,000 14,000,000 16,000,000 18,000,000 20,000,000 22,000,000 24,000,000 26,000,000 28,000,000 30,000,000 32,000,000 34,000,000 36,000,000 38,000,000 40,000,000 42,000,000 44,000,000 46,000,000 48,000,000 50,000,000 52,000,000 54,000,000 56,000,000 58,000,000 60,000,000 62,000,000 64,000,000 66,000,000 68,000,000 70,000,000 72,000,000 74,000,000 76,000,000 78,000,000 80,000,000 82,000,000 84,000,000 86,000,000 88,000,000 90,000,000 92,000,000 94,000,000 96,000,000 98,000,000 100,000,000 

## Pandas

In [1]:
import pandas as pd

df = pd.read_parquet("data/sales")

Der Datensatz wird nun komplett in den Arbeitsspeicher eingelesen:

    Parquet-Dateien
          ↓
       read_parquet
          ↓
    ┌─────────────────┐
    │ kompletter      │
    │ pandas DataFrame│
    │ im Speicher     │
    └─────────────────┘

In [3]:
df.shape

(100000000, 7)

In [5]:
df.head()

,customer_id,product_id,category,country,quantity,price,year,revenue
0,89251,57286,sports,ES,1,70.743790,2024,70.743790
1,773956,84127,electronics,FR,4,162.795670,2023,651.182679
2,654571,21390,sports,FR,2,157.713947,2024,315.427894
3,438879,99774,books,IT,1,433.984144,2025,433.984144
4,433015,7710,electronics,DE,9,135.256105,2024,1217.304942


In [2]:
df.memory_usage()

Index                 132
customer_id     800000000
product_id      800000000
category       1480033604
country        1000000000
quantity        800000000
price           800000000
year            800000000
dtype: int64

In [4]:
df["revenue"] = df["quantity"] * df["price"]

result = (
    df[
        (df["country"] == "DE") &
        (df["year"] == 2025)
    ]
    .groupby("category")["revenue"]
    .mean()
)

print(result)

category
books          1252.924769
clothing       1252.691860
electronics    1251.675527
food           1251.957552
sports         1253.457808
Name: revenue, dtype: float64


## Dask

In [9]:
import dask.dataframe as dd

ddf = dd.read_parquet("data/sales")

Scheinbar das Gleiche, oder?

Aber Dask liest die Daten ja gar nicht direkt ein und v.a. nicht als ein großes Pandas DataFrame.

In [10]:
ddf["revenue"] = ddf["quantity"] * ddf["price"]

result = (
    ddf[
        (ddf["country"] == "DE") &
        (ddf["year"] == 2025)
    ]
    .groupby("category")["revenue"]
    .mean()
)

Bisher ist nichts passiert. Und selbst bei dem folgenden Befehl passiert noch nichts!

In [11]:
print(result)

Dask Series Structure:
npartitions=1
    float64
        ...
Dask Name: getitem, 14 expressions
Expr=(((Filter(frame=Assign(frame=ReadParquetFSSpec(0810202)), predicate=Assign(frame=ReadParquetFSSpec(0810202))['country'] == DE & Assign(frame=ReadParquetFSSpec(0810202))['year'] == 2025))[['category', 'revenue']]).mean(observed=True, chunk_kwargs={'numeric_only': False}, aggregate_kwargs={'numeric_only': False}, _slice='revenue'))['revenue']


Erst mit dem Aufruf von `compute()` passiert etwas:

In [12]:
result = result.compute()

print(result)

category
clothing       1252.691860
books          1252.924769
sports         1253.457808
electronics    1251.675527
food           1251.957552
Name: revenue, dtype: float64


Angenommen unsere 50 Dateien werden zu 50 Partitionen:

    Partition 0 ─┐
    Partition 1 ─┤
    Partition 2 ─┤
    Partition 3 ─┤
                 │
                 ▼
              Scheduler
                 │
         ┌───────┼───────┐
         ▼       ▼       ▼
       Core 1  Core 2  Core 3 ...

Jede Partition ist im Wesentlichen ein pandas-DataFrame.

Dask kann also mehrere Teile parallel bearbeiten.

Du kannst dir die Anzahl der Partitionen ansehen:

In [13]:
print(ddf.npartitions)

50


## PySpark

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("SalesAnalysis")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/21 09:46:33 WARN Utils: Your hostname, juebrauer-Dell-7780, resolves to a loopback address: 127.0.1.1; using 193.174.205.84 instead (on interface enp0s31f6)
26/09/21 09:46:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 09:46:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


local[*] bedeutet hier:

Spark läuft lokal und darf alle verfügbaren CPU-Kerne verwenden.

Dann lesen wir:

In [15]:
sdf = spark.read.parquet("data/sales")

Auch hier bedeutet diese Zeile nicht, dass sofort alle 100 Millionen Zeilen vollständig eingelesen werden.

In [16]:
result = (
    sdf
    .filter(
        (F.col("country") == "DE") &
        (F.col("year") == 2025)
    )
    .withColumn(
        "revenue",
        F.col("quantity") * F.col("price")
    )
    .groupBy("category")
    .agg(
        F.avg("revenue").alias("avg_revenue")
    )
)

Auch das ist zunächst nur ein Plan.

Erst:

    result.show()

führt ihn aus.

In [17]:
result.show()

[Stage 1:===================================>                    (32 + 18) / 50]

+-----------+------------------+
|   category|       avg_revenue|
+-----------+------------------+
|       food| 1251.957551560331|
|      books|1252.9247688492871|
|electronics|  1251.67552736412|
|     sports|1253.4578075894044|
|   clothing|1252.6918600907752|
+-----------+------------------+



Wir können uns den Plan ansehen:

    result.explain()

Oder ausführlicher:

    result.explain("formatted")

In [20]:
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[category#2], functions=[avg(revenue#8)])
   +- Exchange hashpartitioning(category#2, 200), ENSURE_REQUIREMENTS, [plan_id=82]
      +- HashAggregate(keys=[category#2], functions=[partial_avg(revenue#8)])
         +- Project [category#2, (cast(quantity#4L as double) * price#5) AS revenue#8]
            +- Filter (((isnotnull(country#3) AND isnotnull(year#6L)) AND (country#3 = DE)) AND (year#6L = 2025))
               +- FileScan parquet [category#2,country#3,quantity#4L,price#5,year#6L] Batched: true, DataFilters: [isnotnull(country#3), isnotnull(year#6L), (country#3 = DE), (year#6L = 2025)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/media/veracrypt1/07_src/examples/061_parquet_dask_pyspark/data/s..., PartitionFilters: [], PushedFilters: [IsNotNull(country), IsNotNull(year), EqualTo(country,DE), EqualTo(year,2025)], ReadSchema: struct<category:string,country:string,quantity:bigint,price:doubl

In [21]:
result.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [category#2, country#3, quantity#4L, price#5, year#6L]
Batched: true
Location: InMemoryFileIndex [file:/media/veracrypt1/07_src/examples/061_parquet_dask_pyspark/data/sales]
PushedFilters: [IsNotNull(country), IsNotNull(year), EqualTo(country,DE), EqualTo(year,2025)]
ReadSchema: struct<category:string,country:string,quantity:bigint,price:double,year:bigint>

(2) Filter
Input [5]: [category#2, country#3, quantity#4L, price#5, year#6L]
Condition : (((isnotnull(country#3) AND isnotnull(year#6L)) AND (country#3 = DE)) AND (year#6L = 2025))

(3) Project
Output [2]: [category#2, (cast(quantity#4L as double) * price#5) AS revenue#8]
Input [5]: [category#2, country#3, quantity#4L, price#5, year#6L]

(4) HashAggregate
Input [2]: [category#2, revenue#8]
Keys [1]: [category

Da wirst du Dinge sehen wie:

    Scan parquet
        ↓
    Filter
        ↓
    Project
        ↓
    HashAggregate
        ↓
    Exchange
        ↓
    HashAggregate

Das Exchange ist besonders interessant.

Es deutet typischerweise darauf hin, dass Daten zwischen Partitionen umverteilt werden müssen:

    Partition 1       Partition 2
    books             sports
    electronics       books
    sports            electronics

          GROUP BY category
                 ↓

              SHUFFLE

                 ↓

        books       → gemeinsam
        sports      → gemeinsam
        electronics → gemeinsam

Das ist genau einer der Punkte, an denen verteilte Verarbeitung teuer werden kann.

## Fazit des Vergleichs

Man sieht schön den Unterschied:

    pandas      Dask                 Spark
    
    read        read                 read
     ↓           ↓                    ↓
    filter      filter               filter
     ↓           ↓                    ↓
    compute     Task Graph           Logical Plan
                 ↓                    ↓
                compute()            optimizer
                                      ↓
                                     action

# Zugriff auf Zeilen und Spalten bei Spark DataFrames (SDF)

In [32]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkExample") \
    .getOrCreate()

data = [
    ("Anna", 23, "Augsburg"),
    ("Ben", 31, "Bonn"),
    ("Clara", 27, "Chemnitz")
]

sdf = spark.createDataFrame(data, ["name", "age", "city"])

sdf.show()

+-----+---+--------+
| name|age|    city|
+-----+---+--------+
| Anna| 23|Augsburg|
|  Ben| 31|    Bonn|
|Clara| 27|Chemnitz|
+-----+---+--------+



## Spaltenzugriff

In [33]:
# Zugriff auf Spalte age. So ...
sdf["age"]

Column<'age'>

In [34]:
# ... oder so:
from pyspark.sql.functions import col
col("age")

Column<'age'>

Das liefert aber noch nicht die Python-Werte, sondern eine Spark-Column-Expression.

In [35]:
sdf.select("age").show()

+---+
|age|
+---+
| 23|
| 31|
| 27|
+---+



In [36]:
sdf.first()["age"]

23

## Zeilenzugriff

In [37]:
# Für konkrete Zeilenwerte musst du eine Action ausführen, etwa:
rows = sdf.collect()

In [38]:
# rows ist dann eine Python-Liste mit rows-Objekten
rows

[Row(name='Anna', age=23, city='Augsburg'),
 Row(name='Ben', age=31, city='Bonn'),
 Row(name='Clara', age=27, city='Chemnitz')]

In [39]:
# Auf eine Zeile kannst du dann zugreifen:

row = rows[0]

print(row["name"])
print(row["age"])

Anna
23


In [40]:
row.name

'Anna'

In [41]:
row.age

23

In [42]:
# Wenn man nur die ersten 2 Zeilen möchte:
rows = sdf.head(2)

In [43]:
rows

[Row(name='Anna', age=23, city='Augsburg'),
 Row(name='Ben', age=31, city='Bonn')]

Wichtig ist bei Spark auch: Einen allgemeinen Zugriff wie

    sdf.iloc[123]

gibt es beim normalen Spark-DataFrame nicht. **Spark-DataFrames sind verteilte Tabellen und haben nicht dieselbe feste, lokale Zeilenposition wie ein pandas-DataFrame!**

Wenn du also denkst:

    "Ich möchte Zeile 5000"

ist das in Spark normalerweise schon ein Zeichen, dass man das Problem eher über Filter, IDs oder Sortierung formulieren sollte:

    sdf.filter(sdf.customer_id == 12345)
    
statt über eine Zeilennummer.

## Vorsicht bei `collect()`

Und noch ein wichtiger Punkt: Mit

    sdf.collect()

holst du alle Daten vom Cluster in den Python-Prozess des Drivers. Bei großen DataFrames kann das den Speicher sprengen!

Darum arbeitet man eher mit Methoden wie:

In [44]:
sdf.first()

Row(name='Anna', age=23, city='Augsburg')

In [46]:
sdf.head(2)

[Row(name='Anna', age=23, city='Augsburg'),
 Row(name='Ben', age=31, city='Bonn')]

In [47]:
sdf.tail(2)

[Row(name='Ben', age=31, city='Bonn'),
 Row(name='Clara', age=27, city='Chemnitz')]

In [52]:
sdf.take(2)

[Row(name='Anna', age=23, city='Augsburg'),
 Row(name='Ben', age=31, city='Bonn')]

In [53]:
help(sdf.take)

Help on method take in module pyspark.sql.classic.dataframe:

take(num: int) -> List[pyspark.sql.types.Row] method of pyspark.sql.classic.dataframe.DataFrame instance
    Returns the first ``num`` rows as a :class:`list` of :class:`Row`.

    .. versionadded:: 1.3.0

    .. versionchanged:: 3.4.0
        Supports Spark Connect.

    Parameters
    ----------
    num : int
        Number of records to return. Will return this number of records
        or all records if the DataFrame contains less than this number of records..

    Returns
    -------
    list
        List of rows

    Examples
    --------
    >>> df = spark.createDataFrame(
    ...     [(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])

    Return the first 2 rows of the :class:`DataFrame`.

    >>> df.take(2)
    [Row(age=14, name='Tom'), Row(age=23, name='Alice')]



# Gibt es eine erste und letzte Zeile eines Spark DataFrames?

first() bedeutet bei Spark aber nicht „die logisch erste Zeile des Datensatzes“, sondern eher:

    „Gib mir irgendeine Zeile, die in der aktuellen physischen Ausführung zuerst geliefert wird.“

Ein Spark DataFrame ist konzeptionell eine ungeordnete Menge von Zeilen. Ohne orderBy(...) gibt es keine garantierte globale Reihenfolge.

Wenn du also schreibst:

    sdf.first()

bekommst du zwar eine Zeile, aber Spark garantiert dir nicht, dass dieselbe Zeile immer „die erste“ im fachlichen Sinn ist.

Zum Beispiel:

    sdf.orderBy("age").first()

ist dagegen eindeutig: Das ist die Zeile mit dem kleinsten age-Wert.

In [56]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkExample") \
    .getOrCreate()

data = [
    ("Anna", 23, "Augsburg"),
    ("Ben", 31, "Bonn"),
    ("Clara", 27, "Chemnitz")
]

sdf = spark.createDataFrame(data, ["name", "age", "city"])

sdf.show()

+-----+---+--------+
| name|age|    city|
+-----+---+--------+
| Anna| 23|Augsburg|
|  Ben| 31|    Bonn|
|Clara| 27|Chemnitz|
+-----+---+--------+



In [59]:
sdf2 = sdf.orderBy("age", ascending=False)
sdf2.show()

+-----+---+--------+
| name|age|    city|
+-----+---+--------+
|  Ben| 31|    Bonn|
|Clara| 27|Chemnitz|
| Anna| 23|Augsburg|
+-----+---+--------+



# Konvertierung Pandas <-> Spark DataFrame

## SDF -> PDF

In [60]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkExample") \
    .getOrCreate()

data = [
    ("Anna", 23, "Augsburg"),
    ("Ben", 31, "Bonn"),
    ("Clara", 27, "Chemnitz")
]

sdf = spark.createDataFrame(data, ["name", "age", "city"])

sdf.show()

+-----+---+--------+
| name|age|    city|
+-----+---+--------+
| Anna| 23|Augsburg|
|  Ben| 31|    Bonn|
|Clara| 27|Chemnitz|
+-----+---+--------+



In [62]:
pdf = sdf2.toPandas()
pdf

/home/juebrauer/miniconda3/envs/env_teaching/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,name,age,city
0,Ben,31,Bonn
1,Clara,27,Chemnitz
2,Anna,23,Augsburg


Für kleine Ergebnisse ist es sehr praktisch, zum Beispiel nach einer Aggregation:

    result = (
        sdf
        .groupBy("city")
        .avg("age")
    )
    
    pdf = result.toPandas()

Bei einem riesigen DataFrame solltest du dagegen vorher reduzieren:

    pdf = sdf.limit(1000).toPandas()

In [66]:
sdf.limit(1000).toPandas()

/home/juebrauer/miniconda3/envs/env_teaching/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,name,age,city
0,Anna,23,Augsburg
1,Ben,31,Bonn
2,Clara,27,Chemnitz


## PDF -> SDF

In [67]:
import pandas as pd

pdf = pd.DataFrame({
    "name": ["Anna", "Ben", "Clara"],
    "age": [23, 31, 27],
    "city": ["Augsburg", "Bonn", "Chemnitz"]
})
pdf

,name,age,city
0,Anna,23,Augsburg
1,Ben,31,Bonn
2,Clara,27,Chemnitz


In [68]:
sdf = spark.createDataFrame(pdf)
sdf.show()

+-----+---+--------+
| name|age|    city|
+-----+---+--------+
| Anna| 23|Augsburg|
|  Ben| 31|    Bonn|
|Clara| 27|Chemnitz|
+-----+---+--------+



/home/juebrauer/miniconda3/envs/env_teaching/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/juebrauer/miniconda3/envs/env_teaching/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark versucht dabei, das Schema aus den pandas-Datentypen abzuleiten. Du kannst es auch prüfen mit:

    sdf.printSchema()

In [71]:
sdf.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)



# Größe eines SDFs ermitteln

In [74]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkExample") \
    .getOrCreate()

sdf = spark.read.parquet("data/sales")

In [75]:
n_rows = sdf.count()
n_rows

100000000

In [77]:
n_cols = len(sdf.columns)
n_cols

7

In [78]:
sdf

DataFrame[customer_id: bigint, product_id: bigint, category: string, country: string, quantity: bigint, price: double, year: bigint]

Wenn du die ungefähre logische Größe des Spark DataFrames schätzen willst, kannst du dir den optimierten Plan anschauen:

In [79]:
sdf.explain("cost")

== Optimized Logical Plan ==
Relation [customer_id#124L,product_id#125L,category#126,country#127,quantity#128L,price#129,year#130L] parquet, Statistics(sizeInBytes=1676.0 MiB)

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [customer_id#124L,product_id#125L,category#126,country#127,quantity#128L,price#129,year#130L] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/media/veracrypt1/07_src/examples/061_parquet_dask_pyspark/data/s..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:bigint,product_id:bigint,category:string,country:string,quantity:bigint,price:...




In [80]:
# Anzahl der Partitionen:
sdf.rdd.getNumPartitions()

50

# RDDs

## Was ist das?

RDD steht für Resilient Distributed Dataset.

Das war die ursprüngliche zentrale Datenabstraktion von Spark.

Mental kannst du dir Spark so vorstellen:

    Apache Spark
    │
    ├── RDD API
    │
    └── DataFrame / Spark SQL API

Ein RDD ist vereinfacht eine verteilte Sammlung von Objekten. Zum Beispiel:

    rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5])

Dann könntest du schreiben:

    result = (
        rdd
        .filter(lambda x: x % 2 == 0)
        .map(lambda x: x * 10)
    )

    print(result.collect())

Ergebnis:

    [20, 40]

Auch RDDs sind partitioniert und lazy:

    RDD
    ├── Partition 1
    ├── Partition 2
    └── Partition 3

und Operationen wie

    map(...)
    filter(...)

werden zunächst nur als Berechnungsabfolge vorgemerkt.

Erst eine Action wie

    collect()
    count()
    first()

führt die Berechnung aus.

Der große Unterschied zum DataFrame ist: Ein RDD kennt zunächst nur beliebige Objekte, aber kein tabellarisches Schema.

Zum Beispiel:

    rdd = spark.sparkContext.parallelize([
        ("Anna", 23),
        ("Ben", 31)
    ])

Für Spark sind das im Wesentlichen Python-/JVM-Objekte.

Ein DataFrame dagegen:

df = spark.createDataFrame([
    ("Anna", 23),
    ("Ben", 31)
], ["name", "age"])

hat ein Schema:

name: string
age: bigint

Dadurch weiß Spark wesentlich mehr über die Daten und kann besser optimieren.

Das führt zu einem wichtigen Unterschied:

    RDD
    → "Führe diese Funktionen auf meinen Objekten aus."

    DataFrame
    → "Hier ist eine Tabelle mit bekannten Spalten und Datentypen.
       Ich möchte age > 25 filtern."

Bei einem DataFrame kann Spark daher seinen Query Optimizer einsetzen:

    df.filter(df.age > 25)

Spark versteht:

    Filter
    column = age
    condition = > 25
    datatype = bigint

und kann z. B. Predicate Pushdown oder andere Optimierungen durchführen.

Bei einem RDD:

    rdd.filter(lambda x: x[1] > 25)

ist die Operation für Spark viel stärker eine Black Box.

Deshalb arbeitet man heute für tabellarische Daten normalerweise eher mit DataFrames als direkt mit RDDs.

In [82]:
rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5])

In [87]:
type(rdd)

pyspark.core.rdd.RDD

In [84]:
result = (
    rdd
    .filter(lambda x: x % 2 == 0)
    .map(lambda x: x * 10)
)
print(result.collect())

[20, 40]


In [88]:
type(result)

pyspark.core.rdd.PipelinedRDD

In [89]:
rdd = spark.sparkContext.parallelize([
        ("Anna", 23),
        ("Ben", 31)
    ])

## Historische Entwicklung

Die historische Entwicklung lässt sich schön so sehen:

    Spark ursprünglich
          │
          ▼
         RDD
          │
          ▼
    DataFrame
          │
          ▼
    Spark SQL / optimierte strukturierte Verarbeitung

RDD ist aber nicht verschwunden. Es ist weiterhin eine grundlegende Spark-Abstraktion und kann sinnvoll sein, wenn du wirklich mit unstrukturierten oder sehr individuellen Datenstrukturen arbeitest.

## Wann SDFs und wann RDDs?

Der entscheidende Punkt ist:

- DataFrames beschreiben, was du berechnen möchtest.
- RDDs beschreiben stärker, wie du Objekte transformieren möchtest.

Zum Beispiel DataFrame:

    df.filter(df.age > 30)

Spark versteht hier semantisch: Filter auf Spalte age

Beim RDD:

    rdd.filter(lambda x: x.age > 30)

sieht Spark im Wesentlichen nur: führe diese Funktion auf jedem Element aus

und kann viel weniger optimieren.

Deshalb würde ich heute als Faustregel sagen:

    Wenn deine Daten irgendwie sinnvoll als Tabelle darstellbar sind, nimm DataFrames.

RDDs würde ich nur dann bevorzugen, wenn du wirklich unstrukturierte Daten, komplexe Objekte oder einen sehr speziellen Algorithmus hast, der sich schlecht in select, filter, groupBy, join usw. ausdrücken lässt.

## Anwendungsbeispiel: RDDs

Analyse unstrukturierter Daten, z.B. Logdateien, Texte

Wir möchte Fehler (ERROR) pro Modul zählen

In [93]:
from pyspark.sql import SparkSession
from random import randint, choice

LOG_FILE = "data/large_example.log"
N_LINES = 10_000_000


# --------------------------------------------------
# 1. Große Logdatei erzeugen
# --------------------------------------------------
levels = ["INFO", "WARNING", "ERROR"]
modules = ["db", "api", "auth", "cache", "worker"]
messages = [
    "request completed",
    "connection failed",
    "timeout",
    "slow query",
    "invalid token"
]
with open(LOG_FILE, "w") as f:
    for i in range(N_LINES):

        level = choice(levels)
        module = choice(modules)
        message = choice(messages)

        f.write(
            f"2026-09-21 10:{i % 60:02d}:{i % 60:02d} "
            f"{level} "
            f"user={randint(1, 1_000_000)} "
            f"module={module} "
            f'message="{message}"\n'
        )

In [96]:
!head data/large_example.log

2026-09-21 10:00:00 ERROR user=215598 module=cache message="invalid token"
2026-09-21 10:01:01 WARNING user=335428 module=cache message="invalid token"
2026-09-21 10:02:02 WARNING user=652614 module=cache message="timeout"
2026-09-21 10:03:03 WARNING user=494936 module=auth message="request completed"
2026-09-21 10:04:04 INFO user=269382 module=api message="connection failed"
2026-09-21 10:05:05 WARNING user=582803 module=auth message="connection failed"
2026-09-21 10:06:06 WARNING user=613145 module=db message="slow query"
2026-09-21 10:07:07 ERROR user=4339 module=api message="request completed"
2026-09-21 10:08:08 INFO user=417845 module=auth message="connection failed"
2026-09-21 10:09:09 ERROR user=643933 module=api message="timeout"


In [94]:
# --------------------------------------------------
# 2. Spark starten
# --------------------------------------------------
spark = (
    SparkSession.builder
    .appName("RDDLogExample")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext


# --------------------------------------------------
# 3. Logdatei als RDD lesen
# --------------------------------------------------
rdd = sc.textFile(LOG_FILE)


# --------------------------------------------------
# 4. ERROR-Zeilen herausfiltern
# --------------------------------------------------
errors = rdd.filter(
    lambda line: " ERROR " in line
)


# --------------------------------------------------
# 5. Modul extrahieren
# --------------------------------------------------
modules = errors.map(
    lambda line:
        line.split("module=")[1].split()[0]
)


# --------------------------------------------------
# 6. Fehler pro Modul zählen
# --------------------------------------------------
counts = (
    modules
    .map(lambda module: (module, 1))
    .reduceByKey(lambda a, b: a + b)
)


# --------------------------------------------------
# 7. Ergebnis ausgeben
# --------------------------------------------------
print(counts.collect())

spark.stop()

[('auth', 666026), ('api', 667282), ('cache', 666141), ('db', 666540), ('worker', 665525)]


In [98]:
line = '2026-09-21 10:00:00 ERROR user=215598 module=cache message="invalid token"'
line.split("module=")

['2026-09-21 10:00:00 ERROR user=215598 ', 'cache message="invalid token"']

In [101]:
line.split("module=")[1].split()[0]

'cache'

Angenommen modules enthält:

    db
    api
    db
    auth
    db
    api

Dann macht map(...) daraus Schlüssel-Wert-Paare:

    ("db", 1)
    ("api", 1)
    ("db", 1)
    ("auth", 1)
    ("db", 1)
    ("api", 1)

Also sinngemäß:

    Für jedes Modul: „Ich habe dieses Modul einmal gesehen.“

Danach gruppiert reduceByKey(...) gleiche Schlüssel und addiert die Werte:

    db:   1 + 1 + 1 = 3
    api:  1 + 1     = 2
    auth: 1         = 1

Ergebnis:

    ("db", 3)
    ("api", 2)
    ("auth", 1)

reduceByKey(lambda a, b: a + b) bedeutet also schlicht:

    Für jeden Schlüssel alle zugehörigen Werte aufsummieren.